# Session A - system validation

Checks that the loop measures brain and delivers light, before any experimental
session is worth running. Every gate is pre-registered in `app.protocol.quality`
with a fixed threshold, and is reported whether it passes or not.

**G1** railed - **G2** line noise - **G3** alpha blocking - **G4** photic driving -
**G5** instrumentation artifact - **G6** frequency specificity

G3 and G5 are the ones that decide anything. G3 failing means the electrodes are
not on cortex. G5 failing means the strobe is leaking into the amplifier and
every spectral number downstream inherits it.

In [ ]:
import os, sys, json
from pathlib import Path
# Run from the repo root whichever way the notebook was opened, so session
# paths and figure paths mean the same thing.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
%matplotlib inline
import pandas as pd
from app.protocol import analysis as an, figures as fg, quality as q

KIND = "validation"
# Newest matching session. Rehearsals are searched too, for working on the
# analysis before a session has been recorded; they are not published.
ROOTS = [Path("data/sessions"), Path("data/rehearsals")]
candidates = sorted(
    (p for r in ROOTS if r.is_dir() for p in r.glob(f"*_{KIND}*") if p.is_dir()),
    key=lambda p: p.name)

SESSION = candidates[-1] if candidates else None
sess = an.load(SESSION) if SESSION else None
FIG_DIR = Path("artifacts/session_figs") / SESSION.name if SESSION else None
if sess is None:
    print(f"No validation session recorded yet.\\n"
          f"Record one with:  python tools/run_session.py --protocol validation\\n"
          f"Searched: {[str(r) for r in ROOTS]}")
else:
    print(SESSION)


In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    m = sess.manifest
    print(f"protocol   {m['protocol']}")
    print(f"status     {m['status']}   duration {m['duration_s']/60:.1f} min   ticks {m['n_ticks']}")
    print(f"montage    {m.get('montage')}   seed {m.get('seed')}   eeg {m.get('eeg_kind')}")
    print(f"commit     {m.get('git_commit')}   dirty {m.get('git_dirty')}")
    print(f"raw        {None if sess.raw is None else sess.raw.shape}")
    if m.get("abort_reason"): print(f"ABORTED    {m['abort_reason']}")

## Gates

A gate that could not be computed counts as a failure, not as an absence.

The alpha check is **G3 or G8**, not both. G3 needs alpha to rise on eye
closure, which is the weakest it gets inside a mask that is already dark;
G8 asks instead that alpha falls under cognitive load, which does not depend
on the eyes at all. Either one passing shows the electrodes see cortex.


In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    gates = an.validation_gates(sess)
    for g in gates:
        print(g)
        if g.detail: print(f"      {g.detail}")
    summary = q.summarise(gates)
    print()
    alpha = summary["alpha_gate"]
    print(f"alpha check ({alpha['rule']}): "
          + (f"satisfied by {', '.join(alpha['by'])}" if alpha["satisfied"]
             else "NOT satisfied by either route"))
    if summary["warnings"]:
        print(f"non-blocking: {summary['warnings']}")
    print()
    print("PASSED" if summary["passed"]
          else f"BLOCKED BY {summary['blocking_failures']}")


In [ ]:
fg.fig_gate_table(gates)

## Amplitude calibration

Every gate above is a ratio, a decibel or a monotonicity check, so none of
them can see absolute amplitude - and the seven band-power features in the
policy's observation are absolute. This is the number Session B needs and
that only a real recording can supply.


In [ ]:
from tools.calibrate_amplitude import measure
cal = measure(SESSION)
print(f"baseline {cal['block']} ({cal['seconds']:g} s, raw row {cal['raw_row']})")
print(f"measured {cal['measured_rms_uv']:g} uV RMS "
      f"vs {cal['training_rms']:g} in training")
print(f"\n  --amplitude-scale {cal['amplitude_scale']:g}\n")
if not cal["plausible"]:
    print("  ! implausible. A factor this far off is a montage, gain or")
    print("    electrode problem; dividing it out would hide that. Re-run.")


## G3 - alpha blocking

Alpha power must rise when the eyes close, in both repeats. This is the Berger
effect and it is the only direct evidence that the electrodes are picking up
cortex rather than noise.

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    fg.fig_alpha_blocking(sess)

## G4 and G5 - photic driving, and the control that makes it mean something

Solid: the 60 % rung. Dashed: the **0 % brightness** rung - the strobe
still enabled and still commanded at the same frequency, with the light
turned off. Nothing is touched by hand: the LEDs are inside the mask and
the mask never comes off. A peak that survives the dashed line is
electrical pickup from the drivers, not a response to light.


In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    fg.fig_ssvep_and_artifact(sess)

## Save figures for the thesis

In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    written = fg.save_all(sess, FIG_DIR, kind="validation")
    for p in written: print(p)

## Which clock does the mask's LED driver keep?

Not a gate - a setting. Reconstructing the delivered light needs to know
whether the driver's phase carries over when the frequency is retuned. At a
fixed frequency the two models are identical, which is why `A6_ramp_check`
exists. Getting it wrong lowers PLV on both arms equally, so it costs
sensitivity rather than fairness - but PLV is also in the agent's
observation, so a wrong answer is an input it never trained on.


In [ ]:
if sess is None:
    print("no session loaded - see the first cell")
else:
    phase = an.driver_phase_model(sess)
    if not phase["ok"]:
        print("not available:", phase["reason"])
    else:
        print(f"block {phase['block']}  (command sd {phase['command_sd_hz']} Hz)")
        for k, v in phase["plv"].items():
            print(f"  {k:<18} PLV {v:.3f}")
        if phase["decisive"]:
            print(f"\n  -> driver is {phase['best']} (margin {phase['margin']:.3f})")
            if phase["best"] == "restarting":
                print("     set DeliveredEnvelope(phase_continuous=False) for Session B")
        else:
            print(f"\n  the recording does not separate them "
                  f"(margin {phase['margin']:.3f}).")
            print("  That is an answer: the choice is not costing anything here.")
